In [2]:
import pandas as pd

# Caminho onde estão os ficheiros
caminho = "datasets/raw/MQTT/"

print("A carregar e a classificar os dados. Isto pode demorar alguns segundos...")

# Tráfego Normal (Baseline)
df_normal = pd.read_csv(caminho + "biflow_normal.csv")
df_normal['Label_Binario'] = 0       # 0 = Tráfego Seguro
df_normal['Classe'] = 'Normal'

# Tráfego Malicioso (Ataques)
df_bruteforce = pd.read_csv(caminho + "biflow_mqtt_bruteforce.csv")
df_bruteforce['Label_Binario'] = 1   # 1 = Anomalia / Ataque
df_bruteforce['Classe'] = 'Bruteforce_MQTT'

df_scan_A = pd.read_csv(caminho + "biflow_scan_A.csv")
df_scan_A['Label_Binario'] = 1
df_scan_A['Classe'] = 'Scan_A'

df_scan_sU = pd.read_csv(caminho + "biflow_scan_sU.csv")
df_scan_sU['Label_Binario'] = 1
df_scan_sU['Classe'] = 'Scan_UDP'

df_sparta = pd.read_csv(caminho + "biflow_sparta.csv")
df_sparta['Label_Binario'] = 1
df_sparta['Classe'] = 'Sparta_Composto'

# Juntar tudo num único DataFrame
dataset = pd.concat([df_normal, df_bruteforce, df_scan_A, df_scan_sU, df_sparta], ignore_index=True)

print("\nDataset formado com sucesso!")
print(f"Total de pacotes de rede (linhas): {dataset.shape[0]}")
print(f"Total de características (colunas): {dataset.shape[1]}")
print("\nDistribuição exata do tráfego:")
print(dataset['Classe'].value_counts())

A carregar e a classificar os dados. Isto pode demorar alguns segundos...

Dataset formado com sucesso!
Total de pacotes de rede (linhas): 259379
Total de características (colunas): 34

Distribuição exata do tráfego:
Classe
Sparta_Composto    91318
Normal             86008
Scan_UDP           39664
Scan_A             25693
Bruteforce_MQTT    16696
Name: count, dtype: int64


In [3]:
import numpy as np

print("A verificar a integridade do dataset...")

# Encontrar valores infinitos e transformá-los em NaN para podermos limpar tudo de uma vez
dataset = dataset.replace([np.inf, -np.inf], np.nan)

# Contar quantos valores corrompidos temos no total
valores_corrompidos = dataset.isnull().sum().sum()

print(f"Total de valores problemáticos encontrados: {valores_corrompidos}")

if valores_corrompidos > 0:
    # Limpar (apagar as linhas que têm erros)
    dataset_limpo = dataset.dropna()
    print(f"Limpeza concluída! Ficámos com {dataset_limpo.shape[0]} linhas.")
else:
    dataset_limpo = dataset
    print("O dataset já estava limpo! Nenhuma linha foi apagada.")

# Mostrar as colunas para decidirmos o que descartar
print("\nLista de Colunas (Features) disponíveis:")
print(list(dataset_limpo.columns))

A verificar a integridade do dataset...
Total de valores problemáticos encontrados: 0
O dataset já estava limpo! Nenhuma linha foi apagada.

Lista de Colunas (Features) disponíveis:
['ip_src', 'ip_dst', 'prt_src', 'prt_dst', 'proto', 'fwd_num_pkts', 'bwd_num_pkts', 'fwd_mean_iat', 'bwd_mean_iat', 'fwd_std_iat', 'bwd_std_iat', 'fwd_min_iat', 'bwd_min_iat', 'fwd_max_iat', 'bwd_max_iat', 'fwd_mean_pkt_len', 'bwd_mean_pkt_len', 'fwd_std_pkt_len', 'bwd_std_pkt_len', 'fwd_min_pkt_len', 'bwd_min_pkt_len', 'fwd_max_pkt_len', 'bwd_max_pkt_len', 'fwd_num_bytes', 'bwd_num_bytes', 'fwd_num_psh_flags', 'bwd_num_psh_flags', 'fwd_num_rst_flags', 'bwd_num_rst_flags', 'fwd_num_urg_flags', 'bwd_num_urg_flags', 'is_attack', 'Label_Binario', 'Classe']


In [4]:
from sklearn.model_selection import train_test_split

print("A preparar a separação dos dados...")

# 1Isolar as colunas que a IA NÃO deve ver (as "viciadas" e as respostas)
colunas_a_remover = ['ip_src', 'ip_dst', 'prt_src', 'prt_dst', 'proto', 'is_attack', 'Label_Binario', 'Classe']

# X = As perguntas (Características do tráfego)
X = dataset_limpo.drop(columns=colunas_a_remover)

# y = A resposta que queremos prever (Vamos usar a classificação detalhada)
y = dataset_limpo['Classe']

print(f"O modelo vai aprender usando {X.shape[1]} características comportamentais.")

# Dividir em Treino (80%) e Teste (20%)
# O parâmetro 'stratify=y' garante que mantemos a mesma proporção de ataques em ambos os lados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n📊 Divisão concluída com sucesso:")
print(f"Dados de Treino: {X_train.shape[0]} pacotes")
print(f"Dados de Teste: {X_test.shape[0]} pacotes")

A preparar a separação dos dados...
O modelo vai aprender usando 26 características comportamentais.

📊 Divisão concluída com sucesso:
Dados de Treino: 207503 pacotes
Dados de Teste: 51876 pacotes


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import time

print("A normalizar a escala dos dados...")

# Inicializar o normalizador
scaler = StandardScaler()

# O modelo aprende a escala APENAS com os dados de treino
X_train_scaled = scaler.fit_transform(X_train)

# Os dados de teste são transformados com a escala aprendida, sem nunca os "espreitar"
X_test_scaled = scaler.transform(X_test)

print("A iniciar o treino da Random Forest (Isto pode demorar um pouco)...")
inicio = time.time()

# Criar a Inteligência Artificial
# n_jobs=-1 diz ao computador para usar todos os núcleos do processador para ser mais rápido
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Treinar o modelo 
modelo_rf.fit(X_train_scaled, y_train)

fim = time.time()
print(f"Treino concluído em {round(fim - inicio, 2)} segundos!")

print("\nA avaliar o modelo com os Dados de Teste)...")

# Fazer previsões nos dados que o modelo nunca viu
previsoes = modelo_rf.predict(X_test_scaled)

# Mostrar os Resultados
precisao_geral = accuracy_score(y_test, previsoes)
print(f"\nPrecisão Geral do Modelo: {precisao_geral * 100:.2f}%\n")

print("Relatório Detalhado por Tipo de Tráfego:")
print(classification_report(y_test, previsoes))

A normalizar a escala dos dados...
A iniciar o treino da Random Forest (Isto pode demorar um pouco)...
Treino concluído em 11.05 segundos!

A avaliar o modelo com os Dados de Teste)...

Precisão Geral do Modelo: 77.22%

Relatório Detalhado por Tipo de Tráfego:
                 precision    recall  f1-score   support

Bruteforce_MQTT       0.99      0.95      0.97      3339
         Normal       0.68      0.78      0.73     17202
         Scan_A       0.98      0.78      0.87      5138
       Scan_UDP       0.85      0.68      0.76      7933
Sparta_Composto       0.75      0.77      0.76     18264

       accuracy                           0.77     51876
      macro avg       0.85      0.79      0.82     51876
   weighted avg       0.78      0.77      0.77     51876

